# 02. Baseline cho bốn chuỗi dữ liệu

**Mục tiêu:** chạy bốn baseline (Last Value, Moving Average 3 giờ, Seasonal Naive 24 giờ, Seasonal Naive 168 giờ) cho bốn chuỗi **MT_234, MT_217, MT_162, MT_314** trong cùng một notebook.

**Quy trình và logic:** giữ nguyên cách làm từ notebook Baseline gốc:
- Đọc dữ liệu hourly và các tập Train / Validation / Test đã chia.
- Tính `d_train` trên Train; fit scaler chỉ trên Train cho từng chuỗi.
- Dùng `get_common_eval_timestamps` và `predict_baselines`.
- Chọn baseline theo MAE Validation; Test chỉ dùng để báo cáo.
- Phân tích sai số, tạo biểu đồ và xuất kết quả riêng theo từng chuỗi.

**Cách chạy:** đặt notebook trong thư mục `notebooks/` của dự án hoặc chạy từ thư mục gốc, sau đó chạy lần lượt từ trên xuống. Cần có `data/processed/series_hourly.csv` và các module trong `src/`.

In [ ]:
import sys
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from src.baseline import (
    BASELINE_COLUMNS,
    get_common_eval_timestamps,
    predict_baselines,
)
from src.data import (
    load_hourly_data,
    load_splits,
    fit_and_save_scaler,
)
from src.metrics import (
    calculate_d_train,
    evaluate_forecast,
    evaluate_by_group,
)

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

# Chạy cả bốn chuỗi trong một notebook
CHAINS = ["MT_234", "MT_217", "MT_162", "MT_314"]

RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

for c in CHAINS:
    (RESULTS_DIR / c / "tables").mkdir(parents=True, exist_ok=True)
    (RESULTS_DIR / c / "figures").mkdir(parents=True, exist_ok=True)

print("Chuỗi chạy:", CHAINS)

## 1. Đọc dữ liệu và chia tập

Đọc hourly data và các tập đã chia bằng các hàm hiện có trong `src/data.py`. Không thay đổi quy tắc chia dữ liệu của dự án.

In [ ]:
full_df = load_hourly_data(customers=CHAINS)
splits = load_splits(customers=CHAINS)

print(f"Tổng số timestamp: {len(full_df):,}")
print(f"Train: {len(splits['train']):,} | "
      f"Validation: {len(splits['validation']):,} | "
      f"Test: {len(splits['test']):,} giờ")

display(full_df[CHAINS].describe().T.round(3))

## 2. Tính `d_train` và chuẩn hóa

`d_train` được tính trên tập Train riêng cho từng chuỗi để dùng làm mẫu số MASE. Scaler cũng được fit riêng trên Train cho từng chuỗi và lưu dưới `models/scaler_<chuỗi>.pkl`. Baseline vẫn dự báo trên thang đo gốc.

In [ ]:
d_train = {}
scaler_info = []

for c in CHAINS:
    d_train[c] = calculate_d_train(
        splits["train"][c].to_numpy(dtype=float)
    )
    if not np.isfinite(d_train[c]) or d_train[c] <= 0:
        raise ValueError(f"d_train không hợp lệ cho {c}: {d_train[c]}")

    scaler = fit_and_save_scaler(
        train_df=splits["train"],
        customer=c,
        models_dir=str(MODELS_DIR),
    )
    scaler_info.append({
        "customer": c,
        "d_train": d_train[c],
    })

display(pd.DataFrame(scaler_info).set_index("customer").round(6))

## 3. Mốc thời gian đánh giá

Tạo timestamp Validation và Test riêng cho từng chuỗi bằng `get_common_eval_timestamps`. Mốc đánh giá phải nằm trong index tập đích và có đủ lịch sử để tạo các baseline.

In [ ]:
val_ts, test_ts = {}, {}

for c in CHAINS:
    full_series = full_df[c].astype(float)
    val_index = splits["validation"][c].index
    test_index = splits["test"][c].index

    val_ts[c] = get_common_eval_timestamps(full_series, val_index)
    test_ts[c] = get_common_eval_timestamps(full_series, test_index)

    print(f"{c}: Validation={len(val_ts[c]):,}, Test={len(test_ts[c]):,}")

## 4. Dự báo và đánh giá Validation

Chạy bốn baseline cho từng chuỗi. Chọn baseline tốt nhất **chỉ theo MAE Validation**.

In [ ]:
val_pred, val_res, best_model = {}, {}, {}

for c in CHAINS:
    full_series = full_df[c].astype(float)
    pred = predict_baselines(full_series, val_ts[c])
    y_true = pred["y_true"].to_numpy(dtype=float)

    metrics = {}
    for col in BASELINE_COLUMNS:
        y_pred = pred[col].to_numpy(dtype=float)
        if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
            raise ValueError(f"{c}: có NaN/Inf ở {col}")
        metrics[col] = evaluate_forecast(
            y_true, y_pred, d_train=d_train[c]
        )

    res = pd.DataFrame(metrics).T
    res.index.name = "model"
    val_pred[c] = pred
    val_res[c] = res.sort_values("MAE")
    best_model[c] = val_res[c].index[0]

    print(f"\n=== {c}: KẾT QUẢ VALIDATION ===")
    display(val_res[c].round(4))
    print(f"Baseline tốt nhất: {best_model[c]}")

## 5. Đánh giá Test

Test dùng để báo cáo kết quả của cả bốn baseline. Không dùng Test để lựa chọn mô hình; mô hình được chọn vẫn là baseline có MAE thấp nhất trên Validation.

In [ ]:
test_pred, test_res = {}, {}

for c in CHAINS:
    full_series = full_df[c].astype(float)
    pred = predict_baselines(full_series, test_ts[c])
    y_true = pred["y_true"].to_numpy(dtype=float)

    metrics = {}
    for col in BASELINE_COLUMNS:
        y_pred = pred[col].to_numpy(dtype=float)
        if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
            raise ValueError(f"{c}: có NaN/Inf ở Test, cột {col}")
        metrics[col] = evaluate_forecast(
            y_true, y_pred, d_train=d_train[c]
        )

    res = pd.DataFrame(metrics).T
    res.index.name = "model"
    test_pred[c], test_res[c] = pred, res

    print(f"\n=== {c}: KẾT QUẢ TEST ===")
    display(res.round(4))
    print(f"Chọn từ Validation: {best_model[c]} | "
          f"MAE Test={res.loc[best_model[c], 'MAE']:.4f}")

## 6. Phân tích sai số của baseline tốt nhất

Tính sai số theo giờ trong ngày, theo thứ trong tuần và lưu 10 dự báo sai nhiều nhất trên Test, cho từng chuỗi.

In [ ]:
err_hour, err_dow, top10 = {}, {}, {}

for c in CHAINS:
    m = best_model[c]
    df = test_pred[c].copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    err_hour[c] = evaluate_by_group(df, pred_col=m, group_by="hour")
    err_dow[c] = evaluate_by_group(df, pred_col=m, group_by="dayofweek")

    df["abs_error"] = (df["y_true"] - df[m]).abs()
    top10[c] = df.sort_values("abs_error", ascending=False).head(10)

    print(f"\n=== {c}: {m} ===")
    print("Sai số theo giờ:")
    display(err_hour[c])
    print("Sai số theo thứ:")
    display(err_dow[c])
    print("Top 10 sai số lớn nhất:")
    display(top10[c].round(3))

## 7. Biểu đồ

Tạo ba biểu đồ cho từng chuỗi: so sánh MAE/RMSE trên Validation, dự báo 168 giờ đầu Test và MAE theo giờ trong ngày.

In [ ]:
for c in CHAINS:
    m = best_model[c]
    fig_dir = RESULTS_DIR / c / "figures"

    ax = val_res[c][["MAE", "RMSE"]].plot(kind="bar", figsize=(9, 5))
    ax.set(
        title=f"{c}: MAE và RMSE của baseline - Validation",
        xlabel="Baseline",
        ylabel="Sai số",
    )
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(fig_dir / f"baseline_val_comparison_{c}.png", dpi=300)
    plt.show()
    plt.close()

    n = min(168, len(test_pred[c]))
    ts = pd.to_datetime(test_pred[c]["timestamp"])[:n]
    plt.figure(figsize=(14, 5))
    plt.plot(ts, test_pred[c]["y_true"][:n], label="Thực tế", color="black", linewidth=1.5)
    plt.plot(ts, test_pred[c][m][:n], label=f"Dự báo: {m}", color="red", linestyle="--", linewidth=1.2)
    plt.title(f"{c}: dự báo tuần đầu Test - {m}")
    plt.xlabel("Thời gian")
    plt.ylabel("Phụ tải")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_dir / f"baseline_test_timeseries_{c}.png", dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(9, 4))
    plt.bar(err_hour[c]["hour"], err_hour[c]["MAE"], color="skyblue")
    plt.title(f"{c}: MAE theo giờ trong ngày - {m}")
    plt.xlabel("Giờ trong ngày")
    plt.ylabel("MAE")
    plt.xticks(range(24))
    plt.tight_layout()
    plt.savefig(fig_dir / f"baseline_error_by_hour_{c}.png", dpi=300)
    plt.show()
    plt.close()

## 8. Lưu kết quả

Lưu metric, dự báo Validation/Test, dự báo gộp, timestamp đánh giá, bảng sai số và biểu đồ vào thư mục riêng của từng chuỗi để không ghi đè lẫn nhau.

In [ ]:
saved_files = []

for c in CHAINS:
    result_dir = RESULTS_DIR / c
    table_dir = result_dir / "tables"

    val_output = val_pred[c].copy()
    val_output.insert(0, "split", "validation")
    test_output = test_pred[c].copy()
    test_output.insert(0, "split", "test")

    all_predictions = pd.concat(
        [val_output, test_output], ignore_index=True
    )
    eval_timestamps = pd.concat([
        pd.DataFrame({"split": "validation", "timestamp": val_ts[c]}),
        pd.DataFrame({"split": "test", "timestamp": test_ts[c]}),
    ], ignore_index=True)

    files = {
        table_dir / f"baseline_results_{c}.csv": (val_res[c], True),
        table_dir / f"baseline_test_results_{c}.csv": (test_res[c], True),
        table_dir / f"error_by_hour_{c}.csv": (err_hour[c], False),
        table_dir / f"error_by_dayofweek_{c}.csv": (err_dow[c], False),
        table_dir / f"top10_worst_errors_{c}.csv": (top10[c], False),
        result_dir / f"baseline_predictions_validation_{c}.csv": (val_pred[c], False),
        result_dir / f"baseline_predictions_test_{c}.csv": (test_pred[c], False),
        result_dir / f"baseline_predictions_{c}.csv": (all_predictions, False),
        result_dir / f"eval_timestamps_{c}.csv": (eval_timestamps, False),
    }

    for path, (frame, keep_index) in files.items():
        frame.to_csv(path, index=keep_index)
        saved_files.append(path.relative_to(ROOT).as_posix())

print(f"Đã lưu {len(saved_files)} file kết quả.")
print("\n".join(saved_files))

## 9. Tổng hợp kết quả bốn chuỗi

Bảng tổng hợp MAE Validation và Test, đồng thời cho biết baseline được chọn theo Validation. Dùng bảng này để so sánh các chuỗi trong phần báo cáo.

In [ ]:
summary_rows = []

for c in CHAINS:
    row = {
        "customer": c,
        "best_model_validation": best_model[c],
        "best_validation_MAE": val_res[c].loc[best_model[c], "MAE"],
        "selected_model_test_MAE": test_res[c].loc[best_model[c], "MAE"],
    }
    for model_name in BASELINE_COLUMNS:
        row[f"validation_MAE_{model_name}"] = val_res[c].loc[model_name, "MAE"]
        row[f"test_MAE_{model_name}"] = test_res[c].loc[model_name, "MAE"]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("customer")
display(summary_df.round(4))

summary_path = RESULTS_DIR / "baseline_summary_4_series.csv"
summary_df.to_csv(summary_path)
print("Đã lưu bảng tổng hợp:", summary_path.relative_to(ROOT))

## 10. Hoàn thành

Sau khi chạy hết notebook, kiểm tra bảng Validation/Test và thư mục `results/<chuỗi>/`. Notebook chỉ chạy Baseline; chưa thực hiện MLP/LSTM hoặc lựa chọn mô hình học sâu.